# 01 - The frame index

**One job: build `results/frame_index.csv`.**

The archive holds ~15,000 frames on `Z:`, and every question this project asks starts with
"which frames?" - which darks match this light, which gains were actually used, where the ladder
has gaps. Answering that by walking the filesystem each time costs minutes of SMB latency per
question, so it is answered once into a CSV and read from there forever after.

The index is **one row per frame**, and it is built by *reading the pixels*, not by trusting the
folder or the `IMAGETYP` header. That is D18, and it is not paranoia: dark folders in this
archive mix gain and temperature inside a single exposure folder, and a number of flats and
darks were captured under a `Light` subframe type. Capture settings in the header - gain, offset,
exposure, set-temp, achieved temp - are trusted. The type label is evidence, not truth.

**The division of labour** (D35): `astropix` describes *one frame*. The loop over fifteen
thousand of them - what to skip, when to checkpoint, what to print - is orchestration, and it
lives here where you can read it and change it.

| output | written by |
|---|---|
| `results/frame_index.csv` | the scan cell below |

**Cost: roughly 100 minutes** over SMB on a first pass, dominated by opening each frame. It is
incremental and resumable: interrupting it is safe, and re-running reads only what changed.

In [ ]:
# The library is edited while this notebook is open.  Two problems follow, and
# this cell fixes both so that re-running it is always enough.
#
#   1. A kernel started before an edit keeps serving the old module.  autoreload
#      handles the ordinary case.
#   2. autoreload gives up when a module it is tracking no longer exists on disk
#      -- which is exactly what a rename does -- and then silently serves stale
#      code.  So the package is dropped from sys.modules and imported fresh.
%load_ext autoreload
%autoreload 2

import os, pathlib, sys, time
import datetime as dt

sys.path.insert(0, str(pathlib.Path.cwd().parent))

for _stale in [m for m in list(sys.modules) if m.split(".")[0] == "astropix"]:
    del sys.modules[_stale]

import pandas as pd

from astropix import fits as F

# Backstop.  If this still fires after re-running the cell, the kernel is not
# the interpreter printed below -- check the kernel picker.
_required = ("read", "sample_blocks", "capture_settings", "scan_frame",
             "needs_rescan", "stat_row")
_missing = [n for n in _required if not hasattr(F, n)]
assert not _missing, f"wrong or stale astropix at {F.__file__} (missing: {_missing})"

pd.set_option("display.width", 200)

RESULTS = pathlib.Path("..") / "results"
INDEX = RESULTS / "frame_index.csv"

# The four type folders, not their parent: `_by_type` also holds `_canon`,
# frames from a retired camera, with its own bias/dark/flat/light beneath it.
# Naming the four is the cheap guard; `scan_frame` checking INSTRUME on every
# frame is the one that cannot be defeated by a folder rename (D26).
ARCHIVE = pathlib.Path(r"Z:\pix\_astro\raw\_by_type")
ARCHIVE_ROOTS = [ARCHIVE / t for t in ("bias", "dark", "flat", "light")]

PROGRESS = 100      # frames between progress lines
CHECKPOINT = 200    # frames *read* between saves

RESULTS.mkdir(exist_ok=True)
print(f"python  {sys.version.split()[0]}  ({sys.executable})")
print(f"astropix {F.__file__}\n")
for r in ARCHIVE_ROOTS:
    print(f"{'ok     ' if r.is_dir() else 'MISSING'}  {r}")

## What the library does, on one frame

`F.scan_frame` is the whole of the per-frame work: sample a few row-blocks, reduce them to
features per CFA sub-plane (D4), classify from those features, and check the frame came from
this rig. Everything below is a loop around this one call.

In [ ]:
demo = next(p for p in ARCHIVE_ROOTS[1].rglob("*")
            if p.suffix.lower() in F.FITS_SUFFIXES)
rec = F.scan_frame(demo)

print(demo.name, "\n")
for k in ("gain", "exptime", "ccd_temp", "imagetyp", "instrume"):
    print(f"  {k:12s} {rec[k]!r}")
print()
for k in ("level", "sigma", "block_spread", "tail_frac", "clump_frac",
          "mult16_frac", "sat_frac"):
    print(f"  {k:12s} {rec[k]:.6g}")
print()
for k in ("measured_type", "declared_type", "type_agrees", "status"):
    print(f"  {k:12s} {rec[k]!r}")

## Finding the frames

Walking is cheap; reading is not. This pass does no `stat` and opens nothing - it exists to get
a denominator, so the progress report below can say "of how many".

In [ ]:
started = time.monotonic()
paths = sorted(str(p) for root in ARCHIVE_ROOTS for p in root.rglob("*")
               if p.suffix.lower() in F.FITS_SUFFIXES)
print(f"{len(paths)} frames found in {time.monotonic() - started:.1f} s")

previous = {}
if INDEX.exists():
    # dtype=str throughout: `mtime` is compared for exact equality against a
    # repr, and letting pandas infer a float here is precisely the round-trip
    # that would silently make every refresh a full re-read.
    previous = (pd.read_csv(INDEX, dtype=str)
                .set_index("path", drop=False).to_dict("index"))
print(f"{len(previous)} already indexed")

## The scan

Three rules worth reading before it runs, all of them D19:

- **A frame is opened only if it changed.** `F.needs_rescan` compares stored size and mtime
  against the file, and re-reads anything whose last pass did not end `ok`.
- **Rows are never deleted.** A path that has vanished is marked `missing`, so a number
  published months from now stays traceable to the frame it was measured on.
- **The write is atomic.** A temp file plus `os.replace`, so a reader always sees either the
  previous complete index or the new one, never half of one.

**Before running:** the archive must be frozen. New frames belong in `raw\_inbox\`, not in the
archive, until the run finishes.

In [ ]:
HEAD = ["path", "size", "mtime", "indexed_at", "status",
        "measured_type", "declared_type", "type_agrees"]


def write_index(rows):
    """Atomic: a reader sees the old file or the new one, never half of one."""
    df = pd.DataFrame(list(rows.values()))
    df = df[[c for c in HEAD if c in df.columns]
            + [c for c in df.columns if c not in HEAD]]
    tmp = INDEX.with_suffix(".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, INDEX)


def report(done, total, scanned, failed):
    elapsed = time.monotonic() - started
    rate = done / elapsed if elapsed else 0
    eta = (total - done) / rate if rate else 0
    print(f"  {done:>6}/{total} ({done / max(total, 1):5.1%})  "
          f"read {scanned}, skipped {done - scanned}, {failed} unreadable"
          f"  | {elapsed / 60:5.1f} min elapsed, ~{eta / 60:.0f} min left",
          flush=True)


rows = dict(previous)
stamp = dt.datetime.now().isoformat(timespec="seconds")
scanned = failed = 0
started = time.monotonic()

for done, path in enumerate(paths, 1):
    if F.needs_rescan(path, previous.get(path)):
        row = dict(F.stat_row(path), indexed_at=stamp, status="ok")
        try:
            row.update(F.scan_frame(path))
        except Exception as exc:          # truncated, zero-byte, unreadable
            row["status"] = "unreadable: " + type(exc).__name__
            failed += 1
        rows[path] = row
        scanned += 1
        if scanned % CHECKPOINT == 0:
            write_index(rows)
    if done % PROGRESS == 0:
        report(done, len(paths), scanned, failed)

# Never dropped, only marked -- a published constant must stay traceable to the
# frame it was measured on, even after someone reorganises the archive (D19).
here = set(paths)
gone = [p for p, r in rows.items() if p not in here and r.get("status") == "ok"]
for p in gone:
    rows[p].update(status="missing", indexed_at=stamp)

write_index(rows)
report(len(paths), len(paths), scanned, failed)
print(f"\nindex: {len(rows)} rows, {scanned} read this pass, {failed} unreadable, "
      f"{len(gone)} newly missing -> {INDEX}")

## Did it work?

Three questions, in the order in which a wrong answer would hurt most: did everything get read,
does the classifier agree with the labels, and is the bit-shift what we think it is.

In [ ]:
idx = pd.read_csv(INDEX)
ok = idx[idx.status == "ok"]

print(f"{len(idx)} rows, {len(ok)} readable, "
      f"indexed {idx.indexed_at.min()} .. {idx.indexed_at.max()}\n")
print("status:")
print(idx.status.str.split(":").str[0].value_counts().to_string())
print("\nmeasured type vs declared label:")
print(pd.crosstab(ok.declared_type, ok.measured_type, margins=True).to_string())

A disagreement here is not a bug - it is the archive telling you a frame is mislabelled,
and that list is the seed of the archive cleanup (D20). What *would* be a bug is a clean
diagonal with nothing off it: that would mean the classifier is reading the label it was built
to ignore.

In [ ]:
print("fraction of pixel values that are exact multiples of 16:")
print(ok.mult16_frac.agg(["min", "mean", "max"]).to_string())
print("\ngain vs measured type:")
print(pd.crosstab(ok.gain, ok.measured_type, margins=True).to_string())

`mult16_frac = 1.0` everywhere confirms the 12-bit ADC bit-shifted into 16-bit files: one
count in a file is 1/16 of one ADC count, and the fifteen values between are unreachable. That
single fact governs every noise measurement this project will make, and it is why header
`EGAIN` must not be applied to a stored pixel value without dividing by 16 first.

---

**Next.** The index is a map, not a measurement - nothing here says what the sensor *does*. That
needs a real noise estimator and a photon transfer curve, and the order those arrive in is open
(`DECISIONS` D33).